# Evaluation Notebook


In [11]:
import json, os, re
from pathlib import Path
# load evaluation metrics
with open('matcher_eval.json','r',encoding='utf-8') as f:
    metrics = json.load(f)
print('Evaluation metrics:', metrics)

Evaluation metrics: {'MRR@5': 0.12333333333333334, 'Recall@5': 0.13333333333333333, 'misses_sample': [1, 3, 4]}


In [12]:
# Results table for all methods
import json
import pandas as pd
with open('matcher_eval_all.json','r',encoding='utf-8') as f:
    all_metrics = json.load(f)
rows = []
for method, m in all_metrics.items():
    rows.append({'method': method, 'MRR@5': m.get('MRR@5'), 'Recall@5': m.get('Recall@5')})
df = pd.DataFrame(rows).sort_values('MRR@5', ascending=False).reset_index(drop=True)
print(df.to_string(index=False))
# In notebook viewers this will also display nicely:
try:
    from IPython.display import display
    display(df)
except Exception:
    pass

  method    MRR@5  Recall@5
    bm25 0.153333  0.166667
combined 0.123333  0.133333
   tfidf 0.078333  0.100000


,method,MRR@5,Recall@5
0,bm25,0.153333,0.166667
1,combined,0.123333,0.133333
2,tfidf,0.078333,0.100000


In [7]:
# load parsed tenders and profiles
def load_jsonl(path):
    out=[]
    with open(path,'r',encoding='utf-8') as f:
        for line in f:
            if line.strip(): out.append(json.loads(line))
    return out
tenders = load_jsonl('parsers/parsed_data/parsed_tenders.jsonl')
with open('datageneration/data/profiles.json','r',encoding='utf-8') as f:
    profiles = json.load(f)
print(f'Parsed {len(tenders)} tenders and {len(profiles)} profiles')

Parsed 40 tenders and 10 profiles


In [14]:
# build predicted ranking map from summaries filenames
preds = {}
p=Path('summaries')
for fp in sorted(p.glob('profile_*.md')):
    m = re.match(r'profile_(\d+)_tender_(\d+)', fp.name)
    if not m: continue
    pid = int(m.group(1)); tid = int(m.group(2))
    preds.setdefault(pid,[]).append(tid)
# show top-5 for first profile
for pid, ranks in list(preds.items())[:3]:
    print('Profile',pid,'top5:',ranks[:5])

Profile 10 top5: [1, 22, 26, 4, 5]
Profile 1 top5: [1, 12, 2, 37, 38]
Profile 2 top5: [19, 25, 28, 3, 4]


In [10]:
# show confusion cases (misses_sample from matcher_eval.json)
misses = metrics.get('misses_sample', [])
for pid in misses:
    print('--- Profile', pid)
    prof = next((x for x in profiles if x['id']==pid), None)
    print('Profile needs_text:', prof.get('needs_text') if prof else 'n/a')
    top = preds.get(pid, [])[:5]
    print('Predicted top:', top)
    # show candidate tender summaries if present
    for tid in top:
        path = Path('summaries') / f'profile_{pid}_tender_{tid}.md'
        if path.exists():
            print('--- Tender', tid, 'summary:\n', path.read_text(encoding='utf-8'))
        else:
            print('Summary file missing for', tid)

--- Profile 1
Profile needs_text: Looking for funding in agritech sector
Predicted top: [1, 12, 2, 37, 38]
--- Tender 1 summary:
 # Profile 1 — Tender datageneration\data\tenders\tender_1.html

Fintech Innovation Grant 1 correspond au profil — secteur: fintech. Budget: 200000 USD (well matched to the company size). Date limite: 2027-01-28. Mots correspondants: in.

_Score: 0.3000_

--- Tender 12 summary:
 # Profile 1 — Tender datageneration\data\tenders\tender_12.txt

Agritech Innovation Grant 12 correspond au profil — secteur: agritech. Budget: 1000000 USD (may be too large or too small). Date limite: 2026-09-14. Mots correspondants: agritech, in.

_Score: 0.6740_

--- Tender 2 summary:
 # Profile 1 — Tender datageneration\data\tenders\tender_2.html

Agritech Innovation Grant 2 correspond au profil — secteur: agritech. Budget: 5000 USD (may be too large or too small). Date limite: 2027-02-15. Mots correspondants: agritech, for, in, sector.

_Score: 1.0075_

--- Tender 37 summary:
 # P